# Load Model & Tokenizer with 4-Bit Quantization (Slides 7–14)

We load Qwen2.5-3B-Instruct in 4-bit precision (QLoRA). It has a strong multilingual tokenizer for Khmer and runs comfortably on a free GPU.

In [ ]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth for all latest fixes
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [3]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
max_seq_length = 2048

# Load model and tokenizer (replaces AutoModel and AutoProcessor from Slides 9-14)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,        # 4-bit quantization (QLoRA)
    dtype = None,               # Auto-detects float16 or bfloat16
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.2: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

# Load Khmer Dataset, Format Conversations & Apply Chat Template (Slides 16–19)

We load the Khmer dataset from Hugging Face, inspect its column names, convert each row into the "messages" conversation format, and apply the model's chat template (<|im_start|>...<|im_end|>).


In [4]:
from datasets import load_dataset

# 1. Load Raw Khmer Dataset from Hugging Face (Slide 17)
raw_dataset = load_dataset("Pisethan/khmer-instruct-dataset", split="train")

# Check the first sample to see the exact column names (e.g., 'instruction'/'output' or 'prompt'/'response')
print("Sample raw data:", raw_dataset[0])

# 2. Convert to Conversation Format & Apply Chat Template (Slides 18 & 19)
def format_to_chat_template(sample):
    # Automatically grab the prompt and answer keys regardless of column naming
    prompt_text = sample.get("instruction") or sample.get("prompt") or sample.get("question")
    answer_text = sample.get("output") or sample.get("response") or sample.get("answer")

    # Slide 18: Conversation structure
    messages = [
        {"role": "system", "content": "អ្នកគឺជាជំនួយការ AI ដ៏ឆ្លាតវៃ។"}, # "You are a helpful AI assistant."
        {"role": "user", "content": prompt_text},
        {"role": "assistant", "content": answer_text}
    ]

    # Slide 19: Apply Chat Template into a single string
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = False
    )
    return {"text": formatted_text}

# Map the function across the whole dataset
dataset = raw_dataset.map(format_to_chat_template)

# Split 90% for training and 10% for validation loss (Slide 29)
dataset_split = dataset.train_test_split(test_size=0.1, seed=3407)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

# Print a formatted example to verify Slide 19 output
print("\nFormatted Chat Template:\n", train_dataset[0]["text"])

Sample raw data: {'prompt': 'សូមណែនាំអំពីរបៀបសម្រស់នៅពេលជួបគ្នា?', 'response': "ត្រូវនិយាយសួរសុខទុក្ខ ដូចជា 'សួស្តី', និងបង្ហាញអាកប្បកិរិយាគួរសម។"}


Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Formatted Chat Template:
 <|im_start|>system
អ្នកគឺជាជំនួយការ AI ដ៏ឆ្លាតវៃ។<|im_end|>
<|im_start|>user
សូមជួយបង្កើតផែនការមេរៀនសម្រាប់ថ្នាក់ទី២ក្នុងមេរៀនគណិតវិទ្យា?<|im_end|>
<|im_start|>assistant
មេរៀន៖ លេខពី១ដល់១០។ គោលបំណង៖ អោយសិស្សអាចរាប់លេខបាន។ សកម្មភាព៖ ស្តាប់, រៀបលេខ, លេងហ្គេម។<|im_end|>



# Setup LoRA Adapters / PEFT (Slides 22–25)

Freeze the base model and attach trainable low-rank matrices ($A$ and $B$).   

In [5]:
# Add LoRA adapters (Slide 24)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                         # Low-rank dimension (Slide 23 & 24)
    target_modules = [              # Linear layers to adapt (Slide 24 & 25)
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,                # Recommended alpha == r
    lora_dropout = 0,               # 0 is optimized for Unsloth speed
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves 30% more VRAM
    random_state = 3407,
    use_rslora = True,              # Rank-stabilized LoRA (Slide 24)
)

# Check trainable parameters (Slide 24)
model.print_trainable_parameters()

Unsloth 2026.11.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


# Setup SFTConfig & SFTTrainer (Slides 26–27)

Configure the training hyperparameters and pass the dataset to SFTTrainer. (Note: I set report_to="none" below so you can test right away without creating a Weights & Biases account first, or change it to "wandb" as shown on Slide 30).

In [6]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# Trainer Configuration (Slide 26)
args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    num_train_epochs = 1,               # 1 full pass over the Khmer dataset
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 5,
    eval_strategy = "steps",            # Track Validation Loss (Slide 29)
    eval_steps = 20,
    save_steps = 50,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "Qwen2.5-Khmer-Unsloth",
    report_to = "none",                 # Change to "wandb" if using Weights & Biases (Slide 30)
    dataset_text_field = "text",        # Matches the "text" key we created in Cell 3
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
)

# Initialize Trainer (Slide 27)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,        # Used to diagnose overfitting/underfitting (Slide 29)
    args = args,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/450 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


# Start Training & Monitor Loss (Slides 27–31)

Run the fine-tuning loop and watch both Training Loss and Validation Loss decrease.

In [7]:
# Enable model for training and run (Slide 27)
FastLanguageModel.for_training(model)
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 1 | Total steps = 57
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss,Validation Loss
20,0.039541,0.032509
40,0.017712,0.018826
57,0.015138,0.015461


Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in Qwen2.5-Khmer-Unsloth/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in Qwen2.5-Khmer-Unsloth/checkpoint-57/tokenizer_config.json.


# Save Checkpoint & Test Khmer Inference (Slide 33)

Save your trained LoRA adapter weights and test the model with a Khmer prompt.

In [8]:
# 1. Save your trained adapter checkpoint (Slide 33)
model.save_pretrained("Qwen2.5-Khmer-Unsloth/final_checkpoint")
tokenizer.save_pretrained("Qwen2.5-Khmer-Unsloth/final_checkpoint")

# 2. Switch model to inference mode and test it!
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "អ្នកគឺជាជំនួយការ AI ដ៏ឆ្លាតវៃ។"},
    {"role": "user", "content": "តើប្រទេសកម្ពុជាមានរាជធានីឈ្មោះអ្វី? សូមពន្យល់ឱ្យបានខ្លី។"} # "What is the capital of Cambodia? Explain briefly."
]

# Apply chat template with generation prompt (Slide 19)
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 128, temperature = 0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

Unsloth: Restored added_tokens_decoder metadata in Qwen2.5-Khmer-Unsloth/final_checkpoint/tokenizer_config.json.
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
អ្នកគឺជាជំនួយការ AI ដ៏ឆ្លាតវៃ។
user
តើប្រទេសកម្ពុជាមានរាជធានីឈ្មោះអ្វី? សូមពន្យល់ឱ្យបានខ្លី។
assistant
ប្រទេសកម្ពុជាមានរាជធានីអ្វី ONE។
